***

## Co-teaching: Training neural networks with Noisy Labels

**Author**: Debajyoti Majumdar

* In this notebook, we will be preparing the dataset that will be needed for running the Co-teaching experiment.
* We have to use the following datasets:
    - CIFAR 10
    - CIFAR 100
    - Dataset 1 (As provided in Assignment 2 of the IISc course)
* Add synthetic symmetric and asymettric noise on the dataset:
    - Symmetric with noise rates (20%, 50%)
    - Asymmetric (pairwise) with noise rates (45%)
***

In [10]:
import numpy as np
import os
import pickle
from torchvision.datasets import CIFAR10, CIFAR100
from torchvision import transforms
from PIL import Image
from torchvision import datasets

In [ ]:
# load cifar 10 dataset
# save it as jpeg file
# 

def save_dataset(dataset, out_dir):
    for idx in range(len(dataset)):
        img, label = dataset[idx]  # img is PIL Image if no transform; else tensor
        if not isinstance(img, Image.Image):
            # convert tensor to PIL if necessary
            img = transforms.ToPILImage()(img)
        class_name = dataset.classes[label]
        class_dir = os.path.join(out_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        fname = f"img_{idx:06d}.jpg"
        img.save(os.path.join(class_dir, fname), format="JPEG", quality=95)


def main_cifar_10(output_root="cifar10_images"):
    print("Saving CIFAR 10 images locally")
    train = CIFAR10(root="./data", train=True, download=True)
    test  = CIFAR10(root="./data", train=False, download=True)

    train_out = os.path.join(output_root, "train")
    test_out  = os.path.join(output_root, "test")
    os.makedirs(output_root, exist_ok=True)

    print("Saving training images...")
    save_dataset(train, train_out)
    print("Saving test images...")
    save_dataset(test, test_out)
    print("Done.")


def main_cifar_100(output_root="cifar100_images"):
    print("Saving CIFAR 100 images locally")
    train = CIFAR100(root="./data", train=True, download=True)
    test  = CIFAR100(root="./data", train=False, download=True)

    train_out = os.path.join(output_root, "train")
    test_out  = os.path.join(output_root, "test")
    os.makedirs(output_root, exist_ok=True)

    print("Saving training images...")
    save_dataset(train, train_out)
    print("Saving test images...")
    save_dataset(test, test_out)
    print("Done.")

In [ ]:
main_cifar_10()

Saving CIFAR 10 images locally


100%|██████████| 170498071/170498071 [00:24<00:00, 6884250.03it/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified
Saving training images...
Saving test images...
Done.


<function __main__.main_cifar_100(output_root='cifar100_images')>

In [9]:
main_cifar_100()

Saving CIFAR 100 images locally


100%|██████████| 169001437/169001437 [00:16<00:00, 9959058.99it/s] 


Extracting ./data/cifar-100-python.tar.gz to ./data
Files already downloaded and verified
Saving training images...
Saving test images...
Done.


In [11]:
# Now we are testing out how to add symmetric noise
def load_imagefolder_labels(root_dir: str):
    """
    Load an ImageFolder to obtain clean labels and class mapping.
    Returns: dataset (ImageFolder instance), labels (np.array), class_to_idx (dict), idx_to_class (dict)
    """
    ds = datasets.ImageFolder(root=root_dir, transform=transforms.ToTensor())
    # ImageFolder stores class->idx mapping at ds.class_to_idx and targets as ds.targets (list)
    labels = np.array(ds.targets, dtype=int)
    class_to_idx = ds.class_to_idx
    idx_to_class = {v: k for k, v in class_to_idx.items()}
    return ds, labels, class_to_idx, idx_to_class


In [20]:
ds, labels, class_to_idx, idx_to_class = load_imagefolder_labels('./cifar10_images/train')

In [64]:
import shutil
from pathlib import Path
from typing import List
import random

def gather_class_folders(root: Path) -> List[Path]:
    classes = [p for p in sorted(root.iterdir()) if p.is_dir()]
    return classes

def list_files_in_folder(folder: Path) -> List[Path]:
    return [p for p in sorted(folder.iterdir()) if p.is_file()]


def apply_symmetric_noise(noise_rate: float,
                          source_folder: str,
                          target_folder: str):
    # source_folder = Path(source_folder)
    # target_folder = Path(target_folder)
    # copy data from root folder to noisy_data folder
    try:
        print(f"Creating a copy of {source_folder} in {target_folder}...")
        shutil.copytree(Path(source_folder), Path(target_folder))
    except FileExistsError:
        print(f"Folder already exists, deleting it and recreating: {target_folder}")
        shutil.rmtree(Path(target_folder))
        print(f"Creating a copy of {source_folder} in {target_folder}...")
        shutil.copytree(Path(source_folder), Path(target_folder))
        
    # iterate over each folder in the train set
    class_folders = gather_class_folders(Path(target_folder))
    class_name_list = [p.name for p in class_folders]

    # get file_name list
    class_file_mapping = {}
    for class_name in class_name_list:
        file_name_list = list_files_in_folder(Path(target_folder + "/" + class_name))
        class_file_mapping[class_name] = file_name_list


    manifest = {
        "data": "cifar10",
        "source_folder": source_folder,
        "target_folder": target_folder,
        "noise_rate": noise_rate,
        "records": []  # list of dicts: orig_path, orig_class, new_class, new_path, flipped
    }


    for class_name in class_file_mapping.keys():
        file_name_list = class_file_mapping[class_name]
        
        for file_name in file_name_list:
            random_number = random.random()
            change_label = random_number < noise_rate

            if change_label:
                other_classes = [label for label in class_name_list if label != class_name]
                new_class = random.choice(other_classes)
                new_folder = target_folder + "/" + new_class

                manifest["records"].append({
                "orig_path": str(file_name.resolve()),
                "orig_class": class_name,
                "new_class": new_class,
                "new_path": str(new_folder),
                "flipped": True
            })
                print(f"Moving file:{file_name} to target={new_folder}")
                shutil.move(str(file_name), str(new_folder))
                
            else:
                continue
        

        # for each class label (folder), select a handful of images based on the noise rate
        # randomly choose another class for these selected samples
        # for each new noisy label, move the image to that particular folder


In [65]:
apply_symmetric_noise(noise_rate=0.5,
                      source_folder='./cifar10_images/train',
                      target_folder='noise_rate_0.5_cifar10_images')

Creating a copy of ./cifar10_images/train in noise_rate_0.5_cifar10_images...
Folder already exists, deleting it and recreating: noise_rate_0.5_cifar10_images
Creating a copy of ./cifar10_images/train in noise_rate_0.5_cifar10_images...
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000029.jpg to target=noise_rate_0.5_cifar10_images/ship
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000030.jpg to target=noise_rate_0.5_cifar10_images/cat
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000049.jpg to target=noise_rate_0.5_cifar10_images/truck
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000093.jpg to target=noise_rate_0.5_cifar10_images/dog
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000115.jpg to target=noise_rate_0.5_cifar10_images/cat
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000116.jpg to target=noise_rate_0.5_cifar10_images/deer
Moving file:noise_rate_0.5_cifar10_images/airplane/img_000129.jpg to target=noise_rate_0.5_cifar10_

In [ ]:
def apply_asymmteric_noise():
    # take examples from one class and choose examples of another class and switch data samples from it.
    # get the list of folders
    # create a pairwise mapping, indicating the source->target mapping
    ...


In [46]:
list_files_in_folder(Path('./cifar10_images/train/airplane'))[0].resolve()

PosixPath('/Users/deb/PycharmProjects/IISC_course/final_assignment_co_teaching/cifar10_images/train/airplane/img_000029.jpg')

In [35]:
gather_class_folders(Path('./cifar10_images/train/'))[0].name

'airplane'

In [44]:
import random

random.seed(42)
random_number = random.random()
print(random_number)

0.6394267984578837


In [1]:
def asymmetric_noise():
    # TODO: implement pairwise flipping of the data
    ...